# Lab 3 - Explaining Deep Learning Models

**How the lab works**

- The questions are divided into three levels.
  - **[Basic]**: if your group completes all Basic questions of a lab correctly, you get a 6 (a passing grade).
  - **[Regular]**: these questions give up to 2 extra points.
  - **[Advanced]**: for students looking for a challenge; expect to work on them in your own time.
- Every group works with slightly different data and settings, so your answers, numbers and plots are different from those of other groups.
  - Set your group number $G$ in the first code cell. Your seed is $S = G$; use it everywhere a random seed is needed (`random_state=S`).
  - *Your patient* is row $p = S \bmod n$ of your test set, where $n$ is the number of rows in the test set.
  - Put your group number in the title of every plot.
- Answers must use the numbers and plots from your own notebook. An answer that does not match your own output gets no points.
- You may use AI tools to help with code, but every group member must be able to explain what you hand in.
- Lines marked `# TODO` must be completed. Write your answers in the **Your answers** cells.
- Hand in this notebook with all cells run. Also convert the notebook with all cells run into PDF and hand in. 
  
**Data and models.** A pretrained ImageNet model (ResNet-50) and the images in `images/`; Fashion-MNIST; MNIST with a small CNN.
Your group's image, Fashion-MNIST classes, digits and mystery model are given by `assignment(GROUP_NUMBER)`.
The first run downloads the pretrained model and the data sets, which takes a few minutes.

## Setup

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torchvision.transforms as T
from torchvision import datasets
from torchvision.models import resnet50, ResNet50_Weights

from captum.attr import Occlusion, Saliency
from captum.attr import visualization as viz
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
import shap

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegression

from xai_lab_utils import (group_seed, assignment, FASHION_CLASSES, accuracy,
                           load_or_train_mnist_cnn, mnist_loaders, load_mystery_model)

In [ ]:
# Fill in your group number. Everything else depends on it.
GROUP_NUMBER = None  # TODO: your group number, e.g. 7

S = group_seed(GROUP_NUMBER)   # use S everywhere a random seed is needed
print("Group", GROUP_NUMBER, "- seed S =", S)
torch.manual_seed(S)

a = assignment(GROUP_NUMBER)
a

In [ ]:
# Pretrained ImageNet model
weights = ResNet50_Weights.DEFAULT
model = resnet50(weights=weights).eval()
preprocess = weights.transforms()
labels = weights.meta["categories"]
resize_crop = T.Compose([T.Resize(preprocess.resize_size), T.CenterCrop(preprocess.crop_size)])


def load_image(path):
    """Returns the RGB image (224x224, values in [0, 1]) and the model input tensor."""
    img = Image.open(path).convert("RGB")
    rgb = np.asarray(resize_crop(img), dtype=np.float32) / 255
    x = preprocess(img).unsqueeze(0)
    return rgb, x


def top_k(x, k=5):
    with torch.no_grad():
        prob = model(x).softmax(1)[0]
    values, idx = prob.topk(k)
    for v, i in zip(values, idx):
        print(f"{labels[i]:30s} {v:.3f}")
    return idx.tolist()

## Q1) Occlusion

a) **[Basic]** Use the code below to find the areas of importance for your assigned image and one other image from the directory. A running example is given for a basketball image. Use the heatmaps to explain qualitatively how well the pretrained model understands the object in the image.

b) **[Basic]** Explain what is important in the data set, so that what we as humans intuitively use to identify objects can be learnt by the machine. Hint: refer back to the lecture example of a misclassification.

c) **[Advanced]** Change the size of the occlusion window (16, 32, 64 pixels). How do the heatmap and the run time change?

d) **[Advanced]** Occlude with a black patch, a grey patch and a blurred patch. How does the heatmap change? Why does the choice of the patch matter?

> **Hint:** `Occlusion(model).attribute(x, target=idx, sliding_window_shapes=(3, 32, 32), strides=(3, 16, 16), baselines=0)`;
> occlusion is slow on a laptop without a GPU; keep the window at 32 and the stride at 16

In [ ]:
EXAMPLE_IMAGE = "images/basketball.jpg"   # running example

rgb, x = load_image(EXAMPLE_IMAGE)
idx = top_k(x)[0]

occlusion = Occlusion(model)
attr = occlusion.attribute(x, target=idx, sliding_window_shapes=(3, 32, 32),
                           strides=(3, 16, 16), baselines=0)
attr = np.transpose(attr.squeeze(0).detach().numpy(), (1, 2, 0))

viz.visualize_image_attr(attr, rgb, method="blended_heat_map", sign="positive",
                         show_colorbar=True, title=f"Occlusion: {labels[idx]} - group {GROUP_NUMBER}")

# TODO (a): your assigned image a["image"] and one other image

**Your answers:**

a) 

b) 

c) 

d) 

## Q2) Unsupervised explainers

*Note:* this question does not require a deep learning model.

a) **[Basic]** Using PCA, find the number of components that each explain at least 10% of the variance (explained variance ratio $\geq 0.1$). How many components do you need to explain 90% of the variance?

b) **[Basic]** Perform PCA and t-SNE on the Fashion-MNIST data set using the code below.

c) **[Basic]** Look at your two assigned classes. Find test images that are classified as the wrong one of the two, and mark them in the t-SNE plot. Explain what can be interpreted using an unsupervised method like this in explaining an incorrect classification.

d) **[Regular]** Run t-SNE with perplexity 5, 30 and 100. What can and what can you not conclude from a t-SNE plot?

e) **[Advanced]** Show the first four PCA components as $28 \times 28$ images. What does each component capture?

> **Hint:** `pca.explained_variance_ratio_`, `np.cumsum(...)`; `TSNE(n_components=2, perplexity=30, init="pca", random_state=S)`;
> misclassified: `(y_te == c1) & (pred == c2)`

In [ ]:
train = datasets.FashionMNIST("data", train=True, download=True)
test = datasets.FashionMNIST("data", train=False, download=True)

rng = np.random.default_rng(S)
i_train = rng.choice(len(train), 5000, replace=False)
i_test = rng.choice(len(test), 3000, replace=False)
X_tr = train.data.numpy()[i_train].reshape(-1, 784) / 255
y_tr = train.targets.numpy()[i_train]
X_te = test.data.numpy()[i_test].reshape(-1, 784) / 255
y_te = test.targets.numpy()[i_test]

In [ ]:
pca = PCA().fit(X_tr)
# TODO (a): use pca.explained_variance_ratio_ and np.cumsum(...)

Z = PCA(n_components=2).fit_transform(X_tr)
plt.figure(figsize=(7, 6))
sc = plt.scatter(Z[:, 0], Z[:, 1], c=y_tr, cmap="tab10", s=4)
plt.legend(sc.legend_elements()[0], FASHION_CLASSES, fontsize=7)
plt.title(f"PCA - group {GROUP_NUMBER}")
plt.show()

In [ ]:
# t-SNE of the test images (takes about a minute)
E = TSNE(n_components=2, perplexity=30, init="pca", random_state=S).fit_transform(X_te)

# A simple classifier, to find misclassified images
clf = LogisticRegression(max_iter=300).fit(X_tr, y_tr)
pred = clf.predict(X_te)
print("test accuracy:", (pred == y_te).mean())

plt.figure(figsize=(7, 6))
sc = plt.scatter(E[:, 0], E[:, 1], c=y_te, cmap="tab10", s=4)
plt.legend(sc.legend_elements()[0], FASHION_CLASSES, fontsize=7)
plt.title(f"t-SNE - group {GROUP_NUMBER}")
plt.show()

c1, c2 = a["fashion_pair"]
print("Your classes:", a["fashion_pair_names"])
# TODO (c): find test images of c1 predicted as c2 (and the other way around) and mark them in the t-SNE plot

**Your answers:**

a) 

b) 

c) 

d) 

e) 

## Q3) Gradient based explainers

a) **[Basic]** Generate a Grad-CAM heatmap overlay for your assigned image. A running example of an aircraft carrier and a submarine is given.

b) **[Basic]** Explain how, when there are multiple objects in a frame, this could be used to identify each object individually. Show this with Grad-CAM for the top 3 classes of the aircraft carrier and submarine image (or another image with several objects).

c) **[Regular]** Sanity check: make a copy of the model and randomise the weights of the last layer (`fc`), then also of `layer4`. Compute a saliency map and Grad-CAM for your assigned image again. Do the maps change? What does it mean if a map does *not* change?

d) **[Advanced]** Show a saliency map, Integrated Gradients and Grad-CAM side by side for your assigned image. Which one is easiest to understand? Which one shows most detail?

e) **[Advanced]** Remove the most important parts of your image step by step (according to each method) and plot the class probability. Which method finds the parts that matter most for the model?

> **Hint:** `cam = GradCAM(model=model, target_layers=[model.layer4[-1]])`; `heat = cam(input_tensor=x, targets=[ClassifierOutputTarget(idx)])[0]`;
> `Saliency(model).attribute(x, target=idx)`, `captum.attr.IntegratedGradients`;
> random weights: `random_model = copy.deepcopy(model)`; `random_model.fc.reset_parameters()`

In [ ]:
EXAMPLE_GRADCAM_IMAGE = "images/aircraft_carrier.jpg"  # running example

rgb, x = load_image(EXAMPLE_GRADCAM_IMAGE)
idx = top_k(x)[0]

cam = GradCAM(model=model, target_layers=[model.layer4[-1]])
heat = cam(input_tensor=x, targets=[ClassifierOutputTarget(idx)])[0]

plt.imshow(show_cam_on_image(rgb, heat, use_rgb=True))
plt.title(f"Grad-CAM: {labels[idx]} - group {GROUP_NUMBER}")
plt.axis("off")
plt.show()

# TODO (a): your assigned image
# TODO (b): Grad-CAM for the top 3 classes of the image with several objects

In [ ]:
# (c) Sanity check: a copy of the model with random weights in the last layer
random_model = copy.deepcopy(model)
random_model.fc.reset_parameters()
# TODO (c): also randomise layer4, e.g.
# for m in random_model.layer4.modules():
#     if hasattr(m, "reset_parameters"): m.reset_parameters()

saliency = Saliency(model).attribute(x, target=idx).abs().max(dim=1)[0][0]
# TODO (c): compute the saliency map and Grad-CAM for random_model and compare

**Your answers:**

a) 

b) 

c) 

d) 

e) 

## Q4) DeepSHAP

a) **[Basic]** Use the `shap` library and the code below to explain the MNIST model.

b) **[Basic]** Look at your assigned pair of digits. Explain what can be understood about the similarity of these digits as seen by the model, using the red and blue areas in your plots.

c) **[Regular]** Use three different backgrounds: black images, random training images, and images of one digit only. How do the explanations change?

> **Hint:** `shap.DeepExplainer(mnist_model, background)`; `explainer.shap_values(images)`;
> images of one digit: `images[digit_labels == d]`; if `DeepExplainer` fails, try `shap.GradientExplainer`

In [ ]:
mnist_model, mnist_train, mnist_test = load_or_train_mnist_cnn()

images, digit_labels = next(iter(mnist_test))
background = images[:100]
test_images = images[100:110]

explainer = shap.DeepExplainer(mnist_model, background)
shap_values = explainer.shap_values(test_images)

# The layout of shap_values differs between shap versions: make it a list with one array per class
sv = np.array(shap_values)
sv_list = [sv[..., i] for i in range(10)] if sv.shape[-1] == 10 else list(sv)

shap_numpy = [np.moveaxis(s, 1, -1) for s in sv_list]
test_numpy = np.moveaxis(test_images.numpy(), 1, -1)
shap.image_plot(shap_numpy, -test_numpy)

print("Your digits:", a["digit_pair"])
# TODO (b): explain images of your two digits

**Your answers:**

a) 

b) 

c) 

## Q5) The mystery model

a) **[Advanced]** Load your mystery model (code below). It was trained on Fashion-MNIST, but on data with a hidden shortcut, like the snow in the husky/wolf example. Use the methods from this lab to find the shortcut. Which class does it affect? What is the shortcut? Show the evidence, and test your answer by adding the shortcut to images of other classes.

> **Hint:** Grad-CAM on the small model: `GradCAM(model=mystery, target_layers=[mystery.conv_layers[3]])`;
> accuracy per class: compare `pred` and `y` for each class separately

In [ ]:
fm_train, fm_test = mnist_loaders(fashion=True)
mystery = load_mystery_model(a["mystery_id"])
print("clean test accuracy:", accuracy(mystery, fm_test))

# Your analysis here

**Your answers:**

a) 

## Q6) Reflection

a) **[Basic]** Which method from this lab would you trust most for a medical image? Use your own results to explain why.

b) **[Regular]** Assess occlusion, Grad-CAM and DeepSHAP on completeness, comprehensibility and stability.

**Your answers:**

a) 

b) 